# RCA Project: Implementation and Analysis of Logistic Regression

**Dataset:** Breast Cancer Wisconsin Diagnostic Dataset (Scikit-learn)  
**Platform:** Python / Google Colab  
**Algorithm:** Logistic Regression  
**Task:** Binary classification (Malignant vs Benign)

*Educational demonstration only; this is not a medical diagnostic tool.*


## 1. Aim and Objectives
Implement Logistic Regression for binary classification, explore the dataset, train a model, predict classes, and evaluate performance.

## 2. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, classification_report, roc_curve, roc_auc_score)
sns.set_theme(style="whitegrid")


## 3. Load Dataset

In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
# Original target: 0=malignant, 1=benign. Remap positive class to malignant.
df["target"] = (df["target"] == 0).astype(int)
df["diagnosis"] = df["target"].map({1: "Malignant", 0: "Benign"})
print("Dataset loaded successfully!")
print("Shape:", df.shape)
df.head()


## 4. Dataset Overview
Features are measurements derived from digitized images of breast mass samples, such as radius, texture, perimeter, area, and smoothness.

**Target labels in this notebook:** 1 = Malignant, 0 = Benign.


## 5. Dataset Information

In [ ]:
df.info()\n

## 6. Statistical Summary

In [ ]:
df.describe()\n

## 7. Missing Values and Duplicates

In [ ]:
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())


## 8. Target Class Distribution

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="diagnosis")
plt.title("Class Distribution")
plt.xlabel("Diagnosis")
plt.ylabel("Count")
plt.show()
print(df["diagnosis"].value_counts())


## 9. EDA: Feature Scatter Plot

In [ ]:
plt.figure(figsize=(7,5))
sns.scatterplot(data=df, x="mean radius", y="mean texture", hue="diagnosis")
plt.title("Mean Radius vs Mean Texture")
plt.show()


## 10. Correlation Heatmap (Selected Features)

In [ ]:
selected = ["mean radius", "mean texture", "mean perimeter", "mean area",
            "mean smoothness", "target"]
plt.figure(figsize=(8,6))
sns.heatmap(df[selected].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()


## 11. Define Features and Target

In [ ]:
X = df.drop(columns=["target", "diagnosis"])
y = df["target"]
print("X shape:", X.shape)
print("y shape:", y.shape)


## 12. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


## 13. Build and Train Logistic Regression
StandardScaler is fit on the training data inside the Pipeline to avoid test-data leakage.

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(max_iter=5000))
])
model.fit(X_train, y_train)
print("Model trained successfully!")


## 14. Make Predictions

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]  # probability of malignant class
print("Actual first 10:", y_test.iloc[:10].to_list())
print("Predicted first 10:", y_pred[:10])
print("Malignant probabilities:", np.round(y_prob[:10], 3))


## 15. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Benign (0)", "Malignant (1)"],
    yticklabels=["Benign (0)", "Malignant (1)"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()
print("Confusion matrix:\n", cm)


## 16. Evaluation Metrics
Accuracy = overall correctness; Precision = correct malignant predictions among predicted malignant; Recall = malignant cases detected; F1 = harmonic mean of precision and recall.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
auc = roc_auc_score(y_test, y_prob)
metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"],
    "Score": [accuracy, precision, recall, f1, auc]
})
metrics_df


## 17. Classification Report

In [ ]:
print(classification_report(y_test, y_pred,
    target_names=["Benign", "Malignant"], zero_division=0))


## 18. Metric Comparison Chart

In [ ]:
plt.figure(figsize=(8,4))
sns.barplot(data=metrics_df, x="Metric", y="Score")
plt.ylim(0, 1)
plt.title("Logistic Regression Evaluation Metrics")
plt.ylabel("Score (0 to 1)")
plt.xticks(rotation=15)
plt.show()


## 19. ROC Curve and AUC

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC={auc:.3f})")
plt.plot([0,1], [0,1], linestyle="--", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curve")
plt.legend()
plt.show()


## 20. Actual vs Predicted

In [ ]:
comparison = pd.DataFrame({"Actual": y_test.values, "Predicted": y_pred})
comparison["Actual Label"] = comparison["Actual"].map({1:"Malignant", 0:"Benign"})
comparison["Predicted Label"] = comparison["Predicted"].map({1:"Malignant", 0:"Benign"})
comparison.head(15)


## 21. Example Prediction

In [ ]:
sample = X_test.iloc[[0]]
prediction = model.predict(sample)[0]
probability = model.predict_proba(sample)[0, 1]
print("Predicted class:", "Malignant" if prediction == 1 else "Benign")
print(f"Malignant probability: {probability:.3f}")
print("Actual class:", "Malignant" if y_test.iloc[0] == 1 else "Benign")


## 22. Export Results

In [ ]:
results = X_test.copy()
results["Actual"] = y_test.values
results["Predicted"] = y_pred
results["Malignant_Probability"] = y_prob
results.to_csv("logistic_regression_predictions.csv", index=False)
metrics_df.to_csv("logistic_regression_metrics.csv", index=False)
print("CSV files saved.")


## 23. Conclusion
Logistic Regression was applied to classify samples into benign and malignant categories using the dataset's numeric features. The model was evaluated with a confusion matrix, accuracy, precision, recall, F1-score, and ROC-AUC. These metrics and visualizations help assess model performance. This is a classroom demonstration only, not a diagnostic system.


## 24. Viva Questions
1. What is Logistic Regression?
2. Why is it used for classification?
3. What is the sigmoid function?
4. What is binary classification?
5. Why do we split data into train and test sets?
6. Why is feature scaling used?
7. What is a confusion matrix?
8. Define accuracy, precision, recall, and F1-score.
9. What is the ROC curve?
10. What does AUC indicate?
11. What is overfitting?
12. Why is `stratify=y` used?
13. Why is `max_iter` set to 5000?
14. Difference between Linear Regression and Logistic Regression?


## 25. Final Checklist
- [ ] Run notebook top-to-bottom
- [ ] Check all outputs and plots
- [ ] Download notebook with outputs (File → Download → .ipynb)
- [ ] Upload notebook and README to GitHub
- [ ] Upload CSV outputs if needed
